In [ ]:
from google.colab import drive
drive.mount("/content/drive",force_remount=True)

Mounted at /content/drive


In [ ]:
from pathlib import Path

data_dir = Path("/content/drive/MyDrive/Snowho_ML_Challenge/Data")
print("Folder exists:", data_dir.exists())
print("Files:", sorted(p.name for p in data_dir.iterdir()))

Folder exists: True
Files: ['PROBLEM_STATEMENT.md', 'sample_submission.csv', 'starter_notebook.ipynb', 'test_cadences.csv', 'test_centroid.npy', 'test_flux.npy', 'test_stars.csv', 'train_cadences.csv', 'train_centroid.npy', 'train_flux.npy', 'train_stars.csv']


In [ ]:
#validate csv file
import numpy as np
import pandas as pd
from scipy.ndimage import median_filter
from astropy.timeseries import LombScargle, BoxLeastSquares

train_stars=pd.read_csv("/content/drive/MyDrive/Snowho_ML_Challenge/Data/train_stars.csv")
test_stars=pd.read_csv("/content/drive/MyDrive/Snowho_ML_Challenge/Data/test_stars.csv")
train_cadences = pd.read_csv("/content/drive/MyDrive/Snowho_ML_Challenge/Data/train_cadences.csv")
test_cadences = pd.read_csv("/content/drive/MyDrive/Snowho_ML_Challenge/Data/test_cadences.csv")
train_flux = np.load("/content/drive/MyDrive/Snowho_ML_Challenge/Data/train_flux.npy",mmap_mode="r")
train_centroid = np.load("/content/drive/MyDrive/Snowho_ML_Challenge/Data/train_centroid.npy",mmap_mode="r")
test_flux = np.load("/content/drive/MyDrive/Snowho_ML_Challenge/Data/test_flux.npy",mmap_mode="r")
test_centroid = np.load("/content/drive/MyDrive/Snowho_ML_Challenge/Data/test_centroid.npy",mmap_mode="r")

In [ ]:
#Validate the shape
assert train_flux.shape == (18000, 1315)
assert test_flux.shape == (7000, 1315)

assert train_centroid.shape == (18000, 2, 1315)
assert test_centroid.shape == (7000, 2, 1315)

assert len(train_stars) == 18000
assert len(test_stars) == 7000

assert len(train_cadences) == 1315
assert len(test_cadences) == 1315

In [ ]:
#validate columns
required_columns1 = {
    "id", "tmag", "teff", "logg",
    "radius", "contamination",
    "camera", "ccd"
}

assert required_columns1.issubset(train_stars.columns)
assert required_columns1.issubset(test_stars.columns)
required_columns2= {
    "cadence", "time_btjd", "quality"
}
assert required_columns2.issubset(train_cadences.columns)
assert required_columns2.issubset(test_cadences.columns)

assert "label" in train_stars.columns
assert "label" not in test_stars.columns

In [ ]:
#validate labels
expected_labels = {
    "planet",
    "eclipsing_binary",
    "background_binary",
    "starspots",
    "pulsating_star",
    "flare_star",
    "instrument_glitch",
    "quiet_star"
}

assert train_stars["label"].notna().all()

actual_labels = set(train_stars["label"].unique())

assert actual_labels == expected_labels

print(train_stars["label"].value_counts())

label
quiet_star           3624
starspots            2882
eclipsing_binary     2247
planet               2145
pulsating_star       1973
flare_star           1804
instrument_glitch    1721
background_binary    1604
Name: count, dtype: int64


In [ ]:
#validate id
assert train_stars["id"].notna().all()
assert test_stars["id"].notna().all()

assert train_stars["id"].is_unique
assert test_stars["id"].is_unique


In [ ]:
#handle missing values
train_missing_fraction = np.mean(
    np.isnan(train_flux),
    axis=1
)

test_missing_fraction = np.mean(
    np.isnan(test_flux),
    axis=1
)

print("Training missing fraction:")
print(pd.Series(train_missing_fraction).describe())

print("Test missing fraction:")
print(pd.Series(test_missing_fraction).describe())

print("Training flux infinities:",
      np.isinf(train_flux).sum())

print("Test flux infinities:",
      np.isinf(test_flux).sum())

Training missing fraction:
count    1.800000e+04
mean     4.030418e-02
std      6.446412e-15
min      4.030418e-02
25%      4.030418e-02
50%      4.030418e-02
75%      4.030418e-02
max      4.030418e-02
dtype: float64
Test missing fraction:
count    7.000000e+03
mean     4.030418e-02
std      2.283059e-15
min      4.030418e-02
25%      4.030418e-02
50%      4.030418e-02
75%      4.030418e-02
max      4.030418e-02
dtype: float64
Training flux infinities: 0
Test flux infinities: 0


In [ ]:
def extract_flux_features(flux_row):

    # Convert to float so calculations are safe
    flux = np.asarray(flux_row, dtype=np.float64)

    # Treat infinities as invalid measurements
    flux[~np.isfinite(flux)] = np.nan

    missing_fraction = np.mean(~np.isfinite(flux))

    # Normalize using the median of valid measurements
    median = np.nanmedian(flux)

    if not np.isfinite(median) or median == 0:

        return {
            "flux_missing_fraction": missing_fraction,
            "flux_std": np.nan,
            "flux_mad": np.nan,
            "flux_min": np.nan,
            "flux_max": np.nan,
            "flux_skewness": np.nan,
            "flux_kurtosis": np.nan,
            "flux_p01": np.nan,
            "flux_p05": np.nan,
            "flux_p25": np.nan,
            "flux_p75": np.nan,
            "flux_p95": np.nan,
            "flux_p99": np.nan,
            "flux_median_abs_change": np.nan,
            "flux_p95_abs_change": np.nan,
            "bright_outlier_count": np.nan,
            "dim_outlier_count": np.nan,
            "bright_outlier_fraction": np.nan,
            "dim_outlier_fraction": np.nan,
            "max_positive_jump": np.nan,
            "max_negative_jump": np.nan,
            "longest_dim_run": np.nan
        }

    flux = flux / median - 1

    valid = np.isfinite(flux)
    values = flux[valid]

    # Require enough observations for reliable statistics
    if len(values) < 2:
        return {
            "flux_missing_fraction": missing_fraction,
            "flux_std": np.nan,
            "flux_mad": np.nan,
            "flux_min": np.nan,
            "flux_max": np.nan,
            "flux_skewness": np.nan,
            "flux_kurtosis": np.nan,
            "flux_p01": np.nan,
            "flux_p05": np.nan,
            "flux_p25": np.nan,
            "flux_p75": np.nan,
            "flux_p95": np.nan,
            "flux_p99": np.nan,
            "flux_median_abs_change": np.nan,
            "flux_p95_abs_change": np.nan,
            "bright_outlier_count": np.nan,
            "dim_outlier_count": np.nan,
            "bright_outlier_fraction": np.nan,
            "dim_outlier_fraction": np.nan,
            "max_positive_jump": np.nan,
            "max_negative_jump": np.nan,
            "longest_dim_run": np.nan
        }

    center = np.median(values)

    mad = np.median(np.abs(values - center))

    # Robust scale estimate
    robust_scale = 1.4826 * mad

    # Fallback for nearly constant curves
    if robust_scale <= 0:
        robust_scale = np.std(values)

    # If the entire curve is constant, avoid division by zero
    if robust_scale <= 0:
        bright_mask = np.zeros(len(flux), dtype=bool)
        dim_mask = np.zeros(len(flux), dtype=bool)
    else:
        bright_mask = flux > center + 3 * robust_scale
        dim_mask = flux < center - 3 * robust_scale

    bright_count = int(np.sum(bright_mask))
    dim_count = int(np.sum(dim_mask))

    # Consecutive differences only when both observations are valid
    valid_pairs = valid[:-1] & valid[1:]

    differences = np.diff(flux)[valid_pairs]

    if len(differences) > 0:
        median_abs_change = np.median(np.abs(differences))
        p95_abs_change = np.percentile(
            np.abs(differences), 95
        )

        max_positive_jump = max(
            0.0, float(np.max(differences))
        )

        max_negative_jump = min(
            0.0, float(np.min(differences))
        )
    else:
        median_abs_change = np.nan
        p95_abs_change = np.nan
        max_positive_jump = np.nan
        max_negative_jump = np.nan

    # Longest consecutive run of unusually dim observations
    longest_dim_run = 0
    current_run = 0

    for is_dim in dim_mask:
        if is_dim:
            current_run += 1
            longest_dim_run = max(
                longest_dim_run, current_run
            )
        else:
            current_run = 0

    # Standardized third and fourth moments
    std = np.std(values)

    if std > 0:
        z = (values - np.mean(values)) / std

        skewness = np.mean(z ** 3)
        kurtosis = np.mean(z ** 4) - 3
    else:
        skewness = 0.0
        kurtosis = 0.0

    return {
        "flux_missing_fraction": missing_fraction,
        "flux_std": float(std),
        "flux_mad": float(mad),
        "flux_min": float(np.min(values)),
        "flux_max": float(np.max(values)),
        "flux_skewness": float(skewness),
        "flux_kurtosis": float(kurtosis),
        "flux_p01": float(np.percentile(values, 1)),
        "flux_p05": float(np.percentile(values, 5)),
        "flux_p25": float(np.percentile(values, 25)),
        "flux_p75": float(np.percentile(values, 75)),
        "flux_p95": float(np.percentile(values, 95)),
        "flux_p99": float(np.percentile(values, 99)),
        "flux_median_abs_change": float(median_abs_change),
        "flux_p95_abs_change": float(p95_abs_change),
        "bright_outlier_count": bright_count,
        "dim_outlier_count": dim_count,
        "bright_outlier_fraction": bright_count / len(flux),
        "dim_outlier_fraction": dim_count / len(flux),
        "max_positive_jump": max_positive_jump,
        "max_negative_jump": max_negative_jump,
        "longest_dim_run": longest_dim_run
    }

In [ ]:
def extract_centroid_features(centroid_row):

    result = {}

    coordinate_names = ["col", "row"]

    for axis, name in enumerate(coordinate_names):

        values = np.asarray(
            centroid_row[axis],
            dtype=np.float64
        )

        values[~np.isfinite(values)] = np.nan

        valid = np.isfinite(values)

        result[f"{name}_missing_fraction"] = np.mean(~valid)

        valid_values = values[valid]

        if len(valid_values) < 2:
            result[f"{name}_std"] = np.nan
            result[f"{name}_range_95"] = np.nan
            result[f"{name}_median_abs_change"] = np.nan
            result[f"{name}_large_jump_count"] = np.nan
            result[f"{name}_large_jump_fraction"] = np.nan
            continue

        result[f"{name}_std"] = np.std(valid_values)

        result[f"{name}_range_95"] = (
            np.percentile(valid_values, 95)
            - np.percentile(valid_values, 5)
        )

        # Compare consecutive valid observations only
        valid_pairs = valid[:-1] & valid[1:]

        differences = np.diff(values)[valid_pairs]
        absolute_changes = np.abs(differences)

        if len(absolute_changes) == 0:
            result[f"{name}_median_abs_change"] = np.nan
            result[f"{name}_large_jump_count"] = np.nan
            result[f"{name}_large_jump_fraction"] = np.nan
            continue

        median_change = np.median(absolute_changes)

        result[f"{name}_median_abs_change"] = median_change

        # Robust threshold for large position changes
        change_mad = np.median(
            np.abs(absolute_changes - median_change)
        )

        threshold = median_change + 5 * 1.4826 * change_mad

        # If changes are constant, avoid classifying all of them as jumps
        if threshold <= median_change:
            large_jumps = absolute_changes > median_change
        else:
            large_jumps = absolute_changes > threshold

        result[f"{name}_large_jump_count"] = int(
            np.sum(large_jumps)
        )

        result[f"{name}_large_jump_fraction"] = float(
            np.mean(large_jumps)
        )

    return result

In [ ]:
def extract_cadence_features(flux_row, cadence_df):

    flux = np.asarray(flux_row, dtype=np.float64)

    quality = cadence_df["quality"].to_numpy(
        dtype=np.int64
    )

    assert len(flux) == len(quality)

    momentum = (quality & 1) != 0
    scattered = (quality & 2) != 0
    gap = (quality & 4) != 0

    # Normalize flux
    valid = np.isfinite(flux)
    median = np.nanmedian(flux)

    if not np.isfinite(median) or median == 0:
        normalized = np.full(len(flux), np.nan)
    else:
        normalized = flux / median - 1

    valid_pairs = (
        np.isfinite(normalized[:-1])
        & np.isfinite(normalized[1:])
    )

    changes = np.diff(normalized)

    # Use a robust threshold for large flux changes
    if valid_pairs.any():

        abs_changes = np.abs(changes[valid_pairs])

        change_median = np.median(abs_changes)

        change_mad = np.median(
            np.abs(abs_changes - change_median)
        )

        threshold = change_median + 5 * 1.4826 * change_mad

        large_change = np.zeros(len(changes), dtype=bool)

        large_change[valid_pairs] = (
            np.abs(changes[valid_pairs]) > threshold
        )

    else:
        large_change = np.zeros(len(changes), dtype=bool)

    # A change between t-1 and t is associated with cadence t
    event_cadence = np.zeros(len(flux), dtype=bool)

    event_cadence[1:] = large_change

    event_count = np.sum(event_cadence)

    def coincidence_fraction(flag):
        if event_count == 0:
            return 0.0

        return float(
            np.sum(event_cadence & flag) / event_count
        )

    return {
        "momentum_fraction": float(np.mean(momentum)),
        "scattered_light_fraction": float(np.mean(scattered)),
        "data_gap_fraction": float(np.mean(gap)),
        "large_change_count": int(np.sum(large_change)),
        "large_change_momentum_fraction":
            coincidence_fraction(momentum),
        "large_change_scattered_fraction":
            coincidence_fraction(scattered),
        "large_change_gap_fraction":
            coincidence_fraction(gap)
    }

In [ ]:
def prepare_light_curve(flux_row):
    flux = np.asarray(flux_row, dtype=np.float64).copy()

    # Identify valid measurements
    valid = np.isfinite(flux)

    if valid.sum() < 50:
        return None

    # Normalize using the median brightness
    baseline = np.nanmedian(flux)

    if not np.isfinite(baseline) or abs(baseline) < 1e-12:
        return None

    normalized_flux = flux / baseline - 1.0

    # Fill missing points by interpolation.
    # This is for feature calculations only.
    positions = np.arange(len(normalized_flux))
    good = np.isfinite(normalized_flux)

    normalized_flux = np.interp(
        positions,
        positions[good],
        normalized_flux[good]
    )

    return normalized_flux

In [ ]:
def extract_starspot_features(flux_row, time_days):
    features = {
        "flux_scatter_6h": np.nan,
        "flux_scatter_1d": np.nan,
        "flux_scatter_3d": np.nan,
        "short_long_variability_ratio": np.nan,
        "slow_variation_amplitude": np.nan,
        "rotation_period_days": np.nan,
        "periodicity_strength": np.nan,
    }

    flux = prepare_light_curve(flux_row)

    if flux is None:
        return features

    time_days = np.asarray(time_days, dtype=float)

    if len(time_days) != len(flux):
        return features

    valid_time = np.isfinite(time_days)

    if valid_time.sum() < 50:
        return features

    # Assumes time is measured in days and roughly evenly sampled.
    cadence = np.nanmedian(np.diff(time_days))

    if not np.isfinite(cadence) or cadence <= 0:
        return features

    # Rolling-median windows: approximately 6 hours, 1 day and 3 days.
    windows = {
        "6h": max(3, int(round(0.25 / cadence))),
        "1d": max(3, int(round(1.0 / cadence))),
        "3d": max(3, int(round(3.0 / cadence))),
    }

    smoothed = {}

    for name, window in windows.items():
        # Keep the window smaller than the light curve.
        window = min(window, len(flux) - 1)

        if window % 2 == 0:
            window -= 1

        window = max(3, window)

        smooth_curve = median_filter(
            flux,
            size=window,
            mode="nearest"
        )

        smoothed[name] = smooth_curve

        features[f"flux_scatter_{name}"] = float(
            np.std(smooth_curve)
        )

    # Compare original variability with the slower variation.
    original_scatter = np.std(flux)

    slow_scatter = np.std(smoothed["3d"])

    features["short_long_variability_ratio"] = float(
        original_scatter / (slow_scatter + 1e-12)
    )

    # Peak-to-peak amplitude of the slow variation.
    slow_curve = smoothed["3d"]

    features["slow_variation_amplitude"] = float(
        np.percentile(slow_curve, 95)
        - np.percentile(slow_curve, 5)
    )

    # Periodicity search using Lomb-Scargle.
    # Keep only finite time measurements.
    t = time_days[valid_time]
    y = flux[valid_time]

    order = np.argsort(t)
    t = t[order]
    y = y[order]

    # Remove duplicate timestamps.
    unique = np.concatenate(([True], np.diff(t) > 0))
    t = t[unique]
    y = y[unique]

    if len(t) >= 50 and t[-1] > t[0]:
        baseline_days = t[-1] - t[0]

        min_period = 0.5
        max_period = min(15.0, baseline_days / 2)

        if max_period > min_period:
            periods = np.linspace(
                min_period,
                max_period,
                300
            )

            frequencies = 1.0 / periods

            try:
                model = LombScargle(t, y)
                power = model.power(frequencies)

                best_index = np.nanargmax(power)

                features["rotation_period_days"] = float(
                    periods[best_index]
                )

                features["periodicity_strength"] = float(
                    power[best_index]
                )

            except (ValueError, FloatingPointError):
                pass

    return features

In [ ]:
def extract_planet_features(flux_row, time_days):
    features = {
        "transit_period_days": np.nan,
        "transit_depth": np.nan,
        "transit_duration_days": np.nan,
        "transit_strength": np.nan,
        "repeated_transit_count": np.nan,
        "transit_depth_consistency": np.nan,
    }

    flux = prepare_light_curve(flux_row)

    if flux is None:
        return features

    time_days = np.asarray(time_days, dtype=float)

    if len(time_days) != len(flux):
        return features

    valid = np.isfinite(time_days)

    if valid.sum() < 100:
        return features

    t = time_days[valid]
    y = flux[valid]

    order = np.argsort(t)
    t = t[order]
    y = y[order]

    # Remove duplicate timestamps.
    unique = np.concatenate(([True], np.diff(t) > 0))
    t = t[unique]
    y = y[unique]

    if len(t) < 100 or t[-1] <= t[0]:
        return features

    cadence = np.median(np.diff(t))
    baseline_days = t[-1] - t[0]

    if not np.isfinite(cadence) or cadence <= 0:
        return features

    # Remove slow trends while preserving short dips.
    window = max(3, int(round(3.0 / cadence)))
    window = min(window, len(y) - 1)

    if window % 2 == 0:
        window -= 1

    window = max(3, window)

    slow_trend = median_filter(
        y,
        size=window,
        mode="nearest"
    )

    detrended_flux = y - slow_trend

    # BLS requires a useful period range.
    min_period = max(0.5, 5 * cadence)
    max_period = min(15.0, baseline_days / 2)

    if max_period <= min_period:
        return features

    periods = np.linspace(
        min_period,
        max_period,
        100
    )

    durations = np.array([
        2 * cadence,
        3 * cadence,
        4 * cadence,
        6 * cadence,
        8 * cadence,
        0.10,
        0.15,
    ])

    durations = np.unique(
        durations[
            (durations > cadence)
            & (durations < min(0.3, max_period / 5))
        ]
    )

    if len(durations) == 0:
        return features

    try:
        bls = BoxLeastSquares(t, detrended_flux)

        result = bls.power(
            periods,
            durations,
            objective="snr"
        )

        best_index = np.nanargmax(result.power)

        best_period = float(result.period[best_index])
        best_duration = float(result.duration[best_index])
        best_depth = float(result.depth[best_index])
        best_strength = float(result.power[best_index])
        best_t0 = float(result.transit_time[best_index])

        features["transit_period_days"] = best_period
        features["transit_depth"] = best_depth
        features["transit_duration_days"] = best_duration
        features["transit_strength"] = best_strength

        # Estimate the depth of each predicted transit.
        transit_centers = np.arange(
            t[0],
            t[-1] + best_period,
            best_period
        )

        # Keep centers close to the BLS transit phase.
        transit_centers = transit_centers[
            np.abs(
                ((transit_centers - best_t0 + best_period / 2)
                 % best_period) - best_period / 2
            ) < cadence
        ]

        # A more direct approach: calculate predicted events from t0.
        first_n = int(np.floor((t[0] - best_t0) / best_period)) - 1
        last_n = int(np.ceil((t[-1] - best_t0) / best_period)) + 1

        predicted_events = (
            best_t0
            + np.arange(first_n, last_n + 1) * best_period
        )

        event_depths = []

        noise = 1.4826 * np.median(
            np.abs(detrended_flux - np.median(detrended_flux))
        )

        for center in predicted_events:
            in_event = np.abs(t - center) <= best_duration / 2

            if in_event.sum() < 1:
                continue

            event_flux = np.median(detrended_flux[in_event])

            # Count a dip only when it is meaningfully below zero.
            if event_flux < -2 * max(noise, 1e-12):
                event_depths.append(float(-event_flux))

        features["repeated_transit_count"] = len(event_depths)

        if len(event_depths) >= 2:
            mean_depth = np.mean(event_depths)

            features["transit_depth_consistency"] = float(
                np.std(event_depths) / (mean_depth + 1e-12)
            )

    except (ValueError, FloatingPointError, IndexError):
        pass

    return features

In [ ]:
def create_feature_table(
    flux_array,
    centroid_array,
    stars_df,
    cadence_df,
    is_train=False
):
    time_days = cadence_df["time_btjd"].to_numpy(dtype=float)
    rows = []

    for i in range(len(stars_df)):

        # Preserve the original star ID
        row = {
            "id": stars_df.iloc[i]["id"]
        }

        # Flux features
        row.update(
            extract_flux_features(flux_array[i])
        )

        # Centroid features
        row.update(
            extract_centroid_features(centroid_array[i])
        )

        # Cadence features
        row.update(
            extract_cadence_features(
                flux_array[i],
                cadence_df)
            )
        # NEW: Starspot features
        row.update(
            extract_starspot_features(
                flux_array[i],
                time_days)
            )

       # NEW: Planet transit features
        row.update(
           extract_planet_features(
                flux_array[i],
                time_days)
            )

        # Stellar metadata
        for column in [
            "tmag",
            "teff",
            "logg",
            "radius",
            "contamination",
            "camera",
            "ccd"
        ]:
            row[column] = stars_df.iloc[i][column]

        # Training labels only
        if is_train:
            row["label"] = stars_df.iloc[i]["label"]

        rows.append(row)

        if (i + 1) % 1000 == 0:
            print(
                f"Processed {i + 1}/{len(stars_df)} stars"
            )

    return pd.DataFrame(rows)
print(train_cadences.columns.tolist())
print(train_cadences.head())

['cadence', 'time_btjd', 'quality']
   cadence    time_btjd  quality
0        0  2104.272000        2
1        1  2104.292833        2
2        2  2104.313667        2
3        3  2104.334500        2
4        4  2104.355333        2


In [ ]:
train_features = create_feature_table(
    flux_array=train_flux,
    centroid_array=train_centroid,
    stars_df=train_stars,
    cadence_df=train_cadences,
    is_train=True
)

test_features = create_feature_table(
    flux_array=test_flux,
    centroid_array=test_centroid,
    stars_df=test_stars,
    cadence_df=test_cadences,
    is_train=False
)
# test_features = create_feature_table(
#     flux_array=test_flux[:100],
#     centroid_array=test_centroid[:100],
#     stars_df=test_stars.iloc[:100].reset_index(drop=True),
#     cadence_df=test_cadences,
#     is_train=False
# )

print("Training features:", train_features.shape)
print("Test features:", test_features.shape)

Processed 1000/18000 stars
Processed 2000/18000 stars
Processed 3000/18000 stars
Processed 4000/18000 stars
Processed 5000/18000 stars
Processed 6000/18000 stars
Processed 7000/18000 stars
Processed 8000/18000 stars
Processed 9000/18000 stars
Processed 10000/18000 stars
Processed 11000/18000 stars
Processed 12000/18000 stars
Processed 13000/18000 stars
Processed 14000/18000 stars
Processed 15000/18000 stars
Processed 16000/18000 stars
Processed 17000/18000 stars
Processed 18000/18000 stars
Processed 1000/7000 stars
Processed 2000/7000 stars
Processed 3000/7000 stars
Processed 4000/7000 stars
Processed 5000/7000 stars
Processed 6000/7000 stars
Processed 7000/7000 stars
Training features: (18000, 63)
Test features: (7000, 62)


In [ ]:
assert len(train_features) == 18000
assert len(test_features) == 7000

assert (
    train_features["id"].tolist()
    == train_stars["id"].tolist()
)

assert (
    test_features["id"].tolist()
    == test_stars["id"].tolist()
)

assert "label" in train_features.columns
assert "label" not in test_features.columns

In [ ]:
train_numeric = train_features.select_dtypes(
    include=[np.number]
)

test_numeric = test_features.select_dtypes(
    include=[np.number]
)

assert not np.isinf(train_numeric.to_numpy()).any()
assert not np.isinf(test_numeric.to_numpy()).any()

In [ ]:
print("Training missing features:")
print(
    train_features.isnull().sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nTest missing features:")
print(
    test_features.isnull().sum()
    .sort_values(ascending=False)
    .head(15)
)

Training missing features:
transit_depth_consistency    11368
flux_missing_fraction            0
id                               0
flux_mad                         0
flux_min                         0
flux_max                         0
flux_std                         0
flux_kurtosis                    0
flux_p01                         0
flux_p05                         0
flux_p25                         0
flux_p75                         0
flux_p95                         0
flux_p99                         0
flux_skewness                    0
dtype: int64

Test missing features:
transit_depth_consistency    4656
id                              0
flux_std                        0
flux_mad                        0
flux_min                        0
flux_missing_fraction           0
flux_skewness                   0
flux_kurtosis                   0
flux_p01                        0
flux_p05                        0
flux_p25                        0
flux_p75                        0
flu

In [ ]:
from sklearn.model_selection import train_test_split
import os
indices = np.arange(len(train_features))

fit_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=train_features["label"]
)
OUTPUT_DIR= Path("/content/drive/MyDrive/Snowho_ML_Challenge/Featuredata")
os.makedirs(OUTPUT_DIR, exist_ok=True)
assert len(fit_idx) == 14400
assert len(val_idx) == 3600

assert len(np.intersect1d(fit_idx, val_idx)) == 0

assert len(np.unique(
    np.concatenate([fit_idx, val_idx])
)) == len(train_features)

np.save(
    os.path.join(OUTPUT_DIR, "fit_idx.npy"),
    fit_idx
)

np.save(
    os.path.join(OUTPUT_DIR, "val_idx.npy"),
    val_idx
)

print("Split saved successfully.")
print("Random seed: 42")
print("Fit rows:", len(fit_idx))
print("Validation rows:", len(val_idx))

Split saved successfully.
Random seed: 42
Fit rows: 14400
Validation rows: 3600


In [ ]:
train_path = os.path.join(
    OUTPUT_DIR,
    "train_features.csv"
)

test_path = os.path.join(
    OUTPUT_DIR,
    "test_features.csv"
)

train_features.to_csv(
    train_path,
    index=False
)

test_features.to_csv(
    test_path,
    index=False
)

print("Saved:", train_path)
print("Saved:", test_path)

Saved: /content/drive/MyDrive/Snowho_ML_Challenge/Featuredata/train_features.csv
Saved: /content/drive/MyDrive/Snowho_ML_Challenge/Featuredata/test_features.csv


In [ ]:
train_check = pd.read_csv(train_path)
test_check = pd.read_csv(test_path)

assert len(train_check) == 18000
assert len(test_check) == 7000

assert (
    train_check["id"].tolist()
    == train_stars["id"].tolist()
)

assert (
    test_check["id"].tolist()
    == test_stars["id"].tolist()
)

assert train_check.columns.tolist() == train_features.columns.tolist()
assert test_check.columns.tolist() == test_features.columns.tolist()

print("Final saved-file checks passed.")

Final saved-file checks passed.
